# 🔬 Skin Cancer Ensemble Model: Complete Pipeline

This notebook contains the **entire** training, evaluation, prediction, and interpretability pipeline for a Skin Cancer classification project using an ensemble of **ResNet50** and **EfficientNetV2-S**.

### Sections:
1. **Configuration & Imports** — All dependencies and hyperparameters
2. **Data Preparation** — Loading metadata, creating datasets and data loaders
3. **Model Architecture** — Hybrid Ensemble (ResNet50 + EfficientNetV2-S)
4. **Training** — Full training loop with checkpointing
5. **Plot Training History** — Loss and accuracy curves
6. **Model Evaluation** — Classification report, confusion matrix, top-3 accuracy
7. **Single Image Prediction** — Predict on any skin lesion image
8. **Interpretability** — Grad-CAM heatmaps and Saliency Maps

---
## 1. Configuration & Imports

In [ ]:
# Install CPU-only PyTorch (avoids CUDA DLL issues on Windows)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import os
import sys
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import timm
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline

# --- Hyperparameters ---
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/DataScience Project/skin-cancer-mnist-ham10000"
METADATA_PATH = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
MODEL_PATH = "best_model.pth"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-4

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_DESC = {
    'akiec': "Actinic keratoses and intraepithelial carcinoma / Bowen's disease",
    'bcc': 'Basal cell carcinoma',
    'bkl': 'Benign keratosis-like lesions (solar lentigines / seborrheic keratoses and lichen-planus like keratoses)',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic nevi',
    'vasc': 'Vascular lesions (angiomas, angiokeratomas, pyogenic granulomas and hemorrhage)'
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## 2. Data Preparation

In [ ]:
# Copy dataset from Drive to local SSD (run once per Colab session)
# This is the biggest speedup -- Drive I/O is the main bottleneck.
import shutil, os

SRC = DATA_DIR
DST = "/content/data/skin-cancer-mnist-ham10000"

if not os.path.exists(DST):
    print("Copying dataset from Drive to /content/data (~2-3 min, runs once)...")
    shutil.copytree(SRC, DST)
    print("Copy complete.")
else:
    print("Local copy already exists, skipping.")

DATA_DIR      = DST
METADATA_PATH = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
print(f"DATA_DIR is now: {DATA_DIR}")

In [ ]:
# Load metadata and map image paths
print("Loading metadata and mapping images...")
df = pd.read_csv(METADATA_PATH)

# Map image IDs to their file paths (handling both image parts)
image_id_to_path = {os.path.splitext(os.path.basename(x))[0]: x
                     for x in glob(os.path.join(DATA_DIR, '*', '*.jpg'))}

df['path'] = df['image_id'].map(image_id_to_path)
df = df.dropna(subset=['path']).reset_index(drop=True)
df['label'] = pd.Categorical(df['dx']).codes

# Train/Val split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

# Compute class weights for imbalanced data
class_weights = compute_class_weight('balanced', classes=np.unique(df['label']), y=df['label'])
class_weights = torch.tensor(class_weights, dtype=torch.float)

print(f"Total samples: {len(df)}, Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Class weights: {class_weights}")

In [ ]:
# Custom Dataset
class HAM1000Dataset(Dataset):
    def __init__(self, df, transform=None):
        # Drop any rows with missing paths at construction time
        self.df = df[df['path'].notna() & df['path'].apply(lambda p: isinstance(p, str))].reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(self.df.iloc[idx]['label'], dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Transforms (Advanced Data Augmentation)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Data Loaders
train_loader = DataLoader(HAM1000Dataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(HAM1000Dataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

---
## 3. Model Architecture — Hybrid Ensemble

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, num_classes=7):
        super(HybridModel, self).__init__()
        
        # ResNet Branch
        print("Initializing ResNet50...")
        self.resnet = models.resnet50(weights='IMAGENET1K_V1')
        self.resnet_features = nn.Sequential(*list(self.resnet.children())[:-1])
        
        # EfficientNet V2 Branch
        print("Initializing EfficientNetV2-S...")
        self.effnet = timm.create_model('tf_efficientnetv2_s.in1k', pretrained=True, num_classes=0)
        
        # Determine feature size dynamically
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
        with torch.no_grad():
            f1 = self.resnet_features(dummy).view(1, -1).shape[1]
            f2 = self.effnet(dummy).view(1, -1).shape[1]
        
        print(f"Feature dimensions: ResNet={f1}, EffNet={f2}, Combined={f1+f2}")
        
        self.classifier = nn.Sequential(
            nn.Linear(f1 + f2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        feat1 = self.resnet_features(x).view(x.size(0), -1)
        feat2 = self.effnet(x).view(x.size(0), -1)
        fused = torch.cat((feat1, feat2), dim=1)
        return self.classifier(fused)

# Instantiate model and training components
model = HybridModel(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("\nModel ready.")

---
## 4. Training

Run this cell to train the ensemble model. The best model checkpoint will be saved to `best_model.pth` and the training history to `history.json`.

> ⚠️ **Note:** Training 20 epochs on the full dataset may take a long time on CPU. Use a GPU for faster training.

In [ ]:
def train():
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        
        # --- Training Phase ---
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * imgs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        epoch_train_loss = train_loss / len(train_loader.dataset)
        epoch_train_acc = train_correct / train_total
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * imgs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = val_correct / val_total
        
        scheduler.step()
        
        # Logging
        duration = time.time() - start_time
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
        print(f"Duration: {duration:.2f}s")
        
        # Checkpointing
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
            }, MODEL_PATH)
            print(f"*** Best model saved at epoch {epoch+1} ***")
            
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
    
    # Save History
    with open('history.json', 'w') as f:
        json.dump(history, f)
    print("\nTraining history saved to 'history.json'.")
    print("Training Complete!")

# Run training
train()

---
## 5. Plot Training History

Visualize the training/validation loss and validation accuracy over epochs.

In [ ]:
def plot_history(history_path='history.json'):
    if not os.path.exists(history_path):
        print(f"Error: {history_path} not found. Please train the model first.")
        return

    with open(history_path, 'r') as f:
        history = json.load(f)

    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(14, 5))

    # Loss Plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'bo-', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'ro-', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Accuracy Plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'go-', label='Validation Accuracy')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png')
    print("Plot saved to 'training_history.png'.")
    plt.show()

plot_history()

---
## 6. Model Evaluation

Generate a full classification report, overall accuracy, top-3 accuracy, and a confusion matrix heatmap.

In [ ]:
def evaluate():
    # 1. Load Model
    print(f"Loading model from {MODEL_PATH}...")
    eval_model = HybridModel(num_classes=7).to(device)
    
    if not os.path.exists(MODEL_PATH):
        print(f"Error: {MODEL_PATH} not found. Please train the model first.")
        return

    checkpoint = torch.load(MODEL_PATH, map_location=device)
    eval_model.load_state_dict(checkpoint['model_state_dict'])
    eval_model.eval()
    print("Model loaded and set to evaluation mode.")

    # 2. Prepare Data
    eval_loader = DataLoader(HAM1000Dataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False)
    
    # 3. Inference
    y_true = []
    y_pred = []
    y_probs = []

    print("Running inference on validation set...")
    with torch.no_grad():
        for imgs, labels in eval_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = eval_model(imgs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
            y_probs.extend(probs.cpu().numpy())

    # 4. Metrics Calculation
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)
    
    print("\n" + "="*30)
    print("CLASSIFICATION REPORT")
    print("="*30)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    
    acc = accuracy_score(y_true, y_pred)
    print(f"Overall Accuracy: {acc:.4f}")

    # Top-3 Accuracy
    top3_correct = 0
    for i in range(len(y_true)):
        top3_indices = np.argsort(y_probs[i])[-3:]
        if y_true[i] in top3_indices:
            top3_correct += 1
    print(f"Top-3 Accuracy: {top3_correct/len(y_true):.4f}")

    # 5. Confusion Matrix Visualization
    print("\nGenerating Confusion Matrix...")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix: Skin Cancer Ensemble')
    
    plt.savefig('confusion_matrix.png')
    print("Confusion matrix saved to 'confusion_matrix.png'.")
    plt.show()

evaluate()

---
## 7. Single Image Prediction

Provide a path to any skin lesion image and get the model's diagnosis with confidence scores.

In [ ]:
def predict(image_path):
    # 1. Load Model
    if not os.path.exists(MODEL_PATH):
        print(f"Error: {MODEL_PATH} not found. Please train the model first.")
        return

    pred_model = HybridModel(num_classes=7).to(device)
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    pred_model.load_state_dict(checkpoint['model_state_dict'])
    pred_model.eval()

    # 2. Preprocess Image
    try:
        image = Image.open(image_path).convert('RGB')
        input_tensor = val_transform(image).unsqueeze(0).to(device)
    except Exception as e:
        print(f"Error loading image: {e}")
        return

    # 3. Inference
    with torch.no_grad():
        outputs = pred_model(input_tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        conf, pred = torch.max(probs, 0)

    # 4. Results
    print("\n" + "="*40)
    print(f"PREDICTION FOR: {os.path.basename(image_path)}")
    print("="*40)
    print(f"Predicted Class: {CLASS_NAMES[pred]} ({conf*100:.2f}%)")
    print(f"Description: {CLASS_DESC[CLASS_NAMES[pred]]}")
    
    print("\nTop 3 Probabilities:")
    top3_prob, top3_idx = torch.topk(probs, 3)
    for i in range(3):
        print(f"{i+1}. {CLASS_NAMES[top3_idx[i]]}: {top3_prob[i]*100:.2f}%")
    print("="*40)
    
    # Display the image
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(f"Prediction: {CLASS_NAMES[pred]} ({conf*100:.1f}%)")
    plt.axis('off')
    plt.show()

In [ ]:
# === Replace with any image path from your dataset ===
# Example:
# predict("skin-cancer-mnist-ham10000/HAM10000_images_part_1/ISIC_0024306.jpg")

---
## 8. Interpretability — Grad-CAM & Saliency Maps

Visualize **what the model is looking at** when making a diagnosis. This section generates:
- **ResNet Grad-CAM** — Heatmap from the ResNet50 branch
- **EfficientNet Grad-CAM** — Heatmap from the EfficientNetV2-S branch
- **Saliency Map** — Pixel-level sensitivity of the entire ensemble

In [ ]:
class EnsembleInterpretability:
    def __init__(self, model):
        self.model = model
        self.resnet_features = model.resnet_features
        self.effnet = model.effnet
        
        # Hooks for Grad-CAM
        self.resnet_gradients = None
        self.resnet_activations = None
        self.effnet_gradients = None
        self.effnet_activations = None
        
        # Register hooks for the last convolutional layers
        # ResNet: layer4 (last block)
        self.resnet_target_layer = self.resnet_features[-1]
        self.resnet_target_layer.register_forward_hook(self.save_resnet_activation)
        self.resnet_target_layer.register_full_backward_hook(self.save_resnet_gradient)
        
        # EffNet: conv_head
        self.effnet_target_layer = self.effnet.conv_head
        self.effnet_target_layer.register_forward_hook(self.save_effnet_activation)
        self.effnet_target_layer.register_full_backward_hook(self.save_effnet_gradient)

    def save_resnet_activation(self, module, input, output):
        self.resnet_activations = output

    def save_resnet_gradient(self, module, grad_input, grad_output):
        self.resnet_gradients = grad_output[0]

    def save_effnet_activation(self, module, input, output):
        self.effnet_activations = output

    def save_effnet_gradient(self, module, grad_input, grad_output):
        self.effnet_gradients = grad_output[0]

    def generate_gradcam(self, image_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(image_tensor)
        
        if class_idx is None:
            class_idx = torch.argmax(output)
            
        score = output[0, class_idx]
        score.backward()

        # ResNet Grad-CAM
        resnet_grads = self.resnet_gradients
        resnet_acts = self.resnet_activations
        resnet_weights = torch.mean(resnet_grads, dim=(2, 3), keepdim=True)
        resnet_cam = torch.sum(resnet_acts * resnet_weights, dim=1).squeeze()
        resnet_cam = F.relu(resnet_cam)
        
        # EffNet Grad-CAM
        effnet_grads = self.effnet_gradients
        effnet_acts = self.effnet_activations
        effnet_weights = torch.mean(effnet_grads, dim=(2, 3), keepdim=True)
        effnet_cam = torch.sum(effnet_acts * effnet_weights, dim=1).squeeze()
        effnet_cam = F.relu(effnet_cam)

        return resnet_cam, effnet_cam

    def generate_saliency(self, image_tensor):
        image_tensor.requires_grad_()
        outputs = self.model(image_tensor)
        max_scores, _ = torch.max(outputs, 1)
        max_scores.backward()
        
        saliency, _ = torch.max(torch.abs(image_tensor.grad.data), dim=1)
        return saliency.squeeze()

In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).to(device)
    tensor = tensor * std[:, None, None] + mean[:, None, None]
    return tensor.clamp(0, 1)

def plot_interpretability(image_path):
    if not os.path.exists(MODEL_PATH):
        print("Error: No model found. Please train the model first.")
        return

    # Load Model
    interp_model = HybridModel(num_classes=7).to(device)
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    interp_model.load_state_dict(checkpoint['model_state_dict'])
    interp_model.eval()

    # Prep Image
    image = Image.open(image_path).convert('RGB')
    input_tensor = val_transform(image).unsqueeze(0).to(device)
    
    # Initialize Interpretability
    interpreter = EnsembleInterpretability(interp_model)
    
    # Predict
    outputs = interp_model(input_tensor)
    class_idx = torch.argmax(outputs).item()
    confidence = torch.softmax(outputs, dim=1)[0, class_idx].item()
    
    # Generate Heatmaps
    res_cam, eff_cam = interpreter.generate_gradcam(input_tensor, class_idx)
    saliency = interpreter.generate_saliency(input_tensor)

    # Rescale CAMs for display
    res_heatmap = Image.fromarray(res_cam.cpu().detach().numpy()).resize(image.size, Image.BILINEAR)
    eff_heatmap = Image.fromarray(eff_cam.cpu().detach().numpy()).resize(image.size, Image.BILINEAR)
    saliency_heatmap = Image.fromarray(saliency.cpu().detach().numpy()).resize(image.size, Image.BILINEAR)

    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    axes[0].imshow(image)
    axes[0].set_title(f"Original\nPred: {CLASS_NAMES[class_idx]} ({confidence*100:.1f}%)")
    axes[0].axis('off')

    axes[1].imshow(image)
    axes[1].imshow(res_heatmap, cmap='jet', alpha=0.5)
    axes[1].set_title("ResNet Grad-CAM")
    axes[1].axis('off')

    axes[2].imshow(image)
    axes[2].imshow(eff_heatmap, cmap='jet', alpha=0.5)
    axes[2].set_title("EfficientNet Grad-CAM")
    axes[2].axis('off')

    axes[3].imshow(saliency_heatmap, cmap='hot')
    axes[3].set_title("Saliency Map")
    axes[3].axis('off')

    plt.tight_layout()
    output_name = f"interpret_{os.path.basename(image_path)}"
    plt.savefig(output_name)
    print(f"Interpretability visualization saved to {output_name}")
    plt.show()

In [ ]:
# === Replace with any image path from your dataset ===
# Example:
# plot_interpretability("skin-cancer-mnist-ham10000/HAM10000_images_part_1/ISIC_0024306.jpg")